# 03_build_gold_tables — revised for current Silver backfill

This version follows `PROJECT_HANDOFF.md` / `CLAUDE.md`:
- reads `workspace.silver.contract_notices` for CN, not `contract_notices_v2`
- reads `workspace.silver.contract_award_notices_subtype` from the uploaded `02b_parse_notices_backfill_with_subtype.ipynb`
- does **not** try to join CN ↔ CAN for ML
- builds CAN-only ML features plus dashboard tables
- keeps casting in Gold


Permissions:

In [0]:
%sql 
DESCRIBE SCHEMA EXTENDED workspace.gold;

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `diego.gaitancastro@student.ie.edu`;
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `juan.lujan@student.ie.edu`;
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `isha.shah@student.ie.edu`;
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `sebastiao.clemente@student.ie.edu`;

GRANT SELECT ON SCHEMA workspace.gold TO `diego.gaitancastro@student.ie.edu`;
GRANT SELECT ON SCHEMA workspace.gold TO `juan.lujan@student.ie.edu`;
GRANT SELECT ON SCHEMA workspace.gold TO `isha.shah@student.ie.edu`;
GRANT SELECT ON SCHEMA workspace.gold TO `sebastiao.clemente@student.ie.edu`;#

In [0]:
%sql
SHOW GRANTS ON SCHEMA workspace.gold;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG = "workspace"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# CN table is still the standard Silver CN table from the main parser/backfill.
CN_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.contract_notices"

# The uploaded 02b subtype backfill writes this CAN table.
# If you later rename it back to contract_award_notices, change only this line.
CAN_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.contract_award_notices_subtype"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

print("CN source:", CN_TABLE)
print("CAN source:", CAN_TABLE)


In [0]:
def clean_text(col):
    return F.trim(F.regexp_replace(F.coalesce(col.cast("string"), F.lit("")), r"\s+", " "))

def to_double_safe(col):
    # Silver intentionally stores numbers as strings. Cast only here in Gold.
    return (
        F.regexp_replace(
            F.regexp_replace(col.cast("string"), ",", "."),
            r"[^0-9.\-]",
            ""
        ).cast("double")
    )

def to_date_safe(col):
    return F.to_date(F.substring(col.cast("string"), 1, 10), "yyyy-MM-dd")

def table_exists(name):
    try:
        spark.table(name).limit(1).count()
        return True
    except Exception:
        return False

assert table_exists(CAN_TABLE), f"Missing CAN table: {CAN_TABLE}"
# CN is needed for daily_activity and notices_unified. If it is missing, run the full CN+CAN parser first.
assert table_exists(CN_TABLE), f"Missing CN table: {CN_TABLE}"


In [0]:
# Static demo FX rates to EUR.
# Replace with ECB-derived rates before final if exact financial normalisation matters.
fx_rows = [
    ("EUR", 1.0000), ("RON", 0.2000), ("PLN", 0.2300), ("CZK", 0.0400),
    ("HUF", 0.0025), ("BGN", 0.5113), ("DKK", 0.1340), ("SEK", 0.0910),
    ("NOK", 0.0870), ("GBP", 1.1800), ("CHF", 1.0700), ("USD", 0.9200),
]
fx = spark.createDataFrame(fx_rows, ["currency", "eur_rate"])

cn_raw = spark.table(CN_TABLE).filter(F.col("parse_error").isNull())
can_raw = spark.table(CAN_TABLE).filter(F.col("parse_error").isNull())

cn_gold = (
    cn_raw
    .select(
        F.col("notice_id").cast("string").alias("notice_id"),
        to_date_safe(F.col("publication_date")).alias("publication_date"),
        clean_text(F.col("buyer_name")).alias("buyer_name"),
        F.col("buyer_country").cast("string").alias("buyer_country"),
        F.col("buyer_type").cast("string").alias("buyer_type"),
        clean_text(F.col("project_title")).alias("project_title"),
        clean_text(F.col("description")).alias("description"),
        F.col("cpv_code").cast("string").alias("cpv_code"),
        F.substring(F.col("cpv_code").cast("string"), 1, 2).alias("cpv_division"),
        F.col("procedure_type").cast("string").alias("procedure_type"),
        to_double_safe(F.col("estimated_value")).alias("estimated_value"),
        F.upper(F.col("currency").cast("string")).alias("currency"),
        F.col("num_lots").cast("int").alias("num_lots"),
        to_date_safe(F.col("submission_deadline")).alias("submission_deadline"),
        F.col("source_file"),
        F.col("parsed_at"),
        F.col("run_date")
    )
)

can_gold_pre_fx = (
    can_raw
    .select(
        F.col("notice_id").cast("string").alias("notice_id"),
        F.col("notice_subtype").cast("string").alias("notice_subtype"),
        to_date_safe(F.col("publication_date")).alias("publication_date"),
        clean_text(F.col("buyer_name")).alias("buyer_name"),
        F.col("buyer_country").cast("string").alias("buyer_country"),
        F.col("buyer_type").cast("string").alias("buyer_type"),
        clean_text(F.col("project_title")).alias("project_title"),
        F.col("cpv_code").cast("string").alias("cpv_code"),
        F.substring(F.col("cpv_code").cast("string"), 1, 2).alias("cpv_division"),
        F.col("procedure_type").cast("string").alias("procedure_type"),
        F.col("result_code").cast("string").alias("result_code"),
        to_double_safe(F.col("award_value")).alias("award_value"),
        F.upper(F.col("currency").cast("string")).alias("currency"),
        clean_text(F.col("winner_name")).alias("winner_name"),
        F.col("winner_country").cast("string").alias("winner_country"),
        F.col("num_tenders").cast("int").alias("num_tenders"),
        F.col("source_file"),
        F.col("parsed_at"),
        F.col("run_date")
    )
)

can_gold = (
    can_gold_pre_fx
    .join(fx, "currency", "left")
    .withColumn("award_value_eur", F.col("award_value") * F.col("eur_rate"))
)

print("CN rows:", cn_gold.count())
print("CAN rows:", can_gold.count())
display(can_gold.limit(20))


## Gold dashboard / ML tables

In [0]:
AWARD_SUBTYPES = ["29", "30", "31", "32"]

awards_analysis = (
    can_gold
    .filter(F.col("result_code") == "selec-w")
    .filter(F.col("award_value").isNotNull())
    .filter(
        (F.col("award_value_eur") >= 0) &
        (F.col("award_value_eur") <= 5_000_000_000_000)
    )
    .filter(F.col("notice_subtype").isin(AWARD_SUBTYPES))
    .select(
        "notice_id", "notice_subtype", "publication_date",
        "buyer_name", "buyer_country", "buyer_type",
        "project_title", "cpv_code", "cpv_division", "procedure_type",
        "award_value", "award_value_eur", "currency",
        "winner_name", "winner_country", "num_tenders",
        "source_file", "run_date"
    )
    .dropDuplicates(["notice_id"])
)

(awards_analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.awards_analysis"))

print("Wrote awards_analysis:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.awards_analysis").count())


In [0]:
ml_features = (
    awards_analysis
    .filter(F.col("award_value").isNotNull())
    .filter(F.col("num_tenders").isNotNull())
    .withColumn("log_award_value", F.log1p(F.col("award_value")))
    .withColumn("log_award_value_eur", F.log1p(F.col("award_value_eur")))
    .withColumn(
        "is_cross_border",
        F.when(
            F.col("buyer_country").isNotNull() & F.col("winner_country").isNotNull(),
            F.col("buyer_country") != F.col("winner_country")
        ).otherwise(F.lit(False))
    )
    .withColumn("is_single_bidder", F.col("num_tenders") == F.lit(1))
    .select(
        "notice_id", "publication_date",
        "buyer_name", "buyer_country", "buyer_type",
        "project_title", "cpv_code", "cpv_division", "procedure_type",
        "award_value", "award_value_eur", "currency",
        "log_award_value", "log_award_value_eur",
        "winner_name", "winner_country", "num_tenders",
        "is_cross_border", "is_single_bidder"
    )
)

(ml_features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.ml_features"))

print("Wrote ml_features:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.ml_features").count())


In [0]:
buyer_base = (
    ml_features
    .groupBy("buyer_name", "buyer_country", "buyer_type")
    .agg(
        F.count("*").alias("total_contracts"),
        F.sum("award_value_eur").alias("total_awarded_value_eur"),
        F.avg("award_value_eur").alias("avg_award_value_eur"),
        F.countDistinct("cpv_code").alias("distinct_cpv_codes"),
        F.max("publication_date").alias("last_tender_date"),
        F.avg(F.col("is_single_bidder").cast("double")).alias("single_bidder_rate"),
        F.avg(F.col("is_cross_border").cast("double")).alias("cross_border_rate")
    )
)

buyer_cpv_counts = (
    ml_features
    .groupBy("buyer_name", "buyer_country", "buyer_type", "cpv_division")
    .agg(F.count("*").alias("cpv_contracts"))
)

w = Window.partitionBy("buyer_name", "buyer_country", "buyer_type").orderBy(F.desc("cpv_contracts"), F.asc("cpv_division"))

top_cpv = (
    buyer_cpv_counts
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("buyer_name", "buyer_country", "buyer_type", F.col("cpv_division").alias("top_cpv_division"))
)

buyer_profiles = buyer_base.join(top_cpv, ["buyer_name", "buyer_country", "buyer_type"], "left")

(buyer_profiles.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.buyer_profiles"))

print("Wrote buyer_profiles:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.buyer_profiles").count())


In [0]:
cpv_summary = (
    awards_analysis
    .groupBy("cpv_division")
    .agg(
        F.count("*").alias("contract_count"),
        F.sum("award_value_eur").alias("total_award_value_eur"),
        F.avg("award_value_eur").alias("avg_award_value_eur"),
        F.countDistinct("buyer_name").alias("distinct_buyers"),
        F.countDistinct("buyer_country").alias("distinct_buyer_countries"),
        F.avg((F.col("num_tenders") == 1).cast("double")).alias("single_bidder_rate")
    )
)

(cpv_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.cpv_summary"))

cn_daily = (
    cn_gold
    .groupBy(F.col("publication_date").alias("activity_date"))
    .agg(
        F.count("*").alias("cn_count"),
        F.sum(F.when(F.col("cpv_division").isin("30", "32", "48", "72"), 1).otherwise(0)).alias("cn_it_count")
    )
)

can_daily = (
    awards_analysis
    .groupBy(F.col("publication_date").alias("activity_date"))
    .agg(
        F.count("*").alias("can_count"),
        F.sum(F.when(F.col("cpv_division").isin("30", "32", "48", "72"), 1).otherwise(0)).alias("can_it_count"),
        F.sum("award_value_eur").alias("total_award_value_eur")
    )
)

daily_activity = (
    cn_daily
    .join(can_daily, "activity_date", "full")
    .fillna(0, ["cn_count", "cn_it_count", "can_count", "can_it_count", "total_award_value_eur"])
    .orderBy("activity_date")
)

(daily_activity.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.daily_activity"))

print("Wrote cpv_summary:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.cpv_summary").count())
print("Wrote daily_activity:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.daily_activity").count())


## Chatbot-friendly unified table

In [0]:
notices_unified = (
    cn_gold.select(
        F.lit("CN").alias("notice_kind"),
        "notice_id", "publication_date", "buyer_name", "buyer_country", "buyer_type",
        "project_title", "description", "cpv_code", "cpv_division", "procedure_type",
        F.col("estimated_value").alias("amount"),
        "currency",
        "submission_deadline",
        F.lit(None).cast("string").alias("result_code"),
        F.lit(None).cast("string").alias("winner_name"),
        F.lit(None).cast("string").alias("winner_country"),
        F.lit(None).cast("int").alias("num_tenders"),
        "source_file", "run_date"
    )
    .unionByName(
        can_gold.select(
            F.lit("CAN").alias("notice_kind"),
            "notice_id", "publication_date", "buyer_name", "buyer_country", "buyer_type",
            "project_title",
            F.lit(None).cast("string").alias("description"),
            "cpv_code", "cpv_division", "procedure_type",
            F.col("award_value").alias("amount"),
            "currency",
            F.lit(None).cast("date").alias("submission_deadline"),
            "result_code", "winner_name", "winner_country", "num_tenders",
            "source_file", "run_date"
        )
    )
)

(notices_unified.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.notices_unified"))

print("Wrote notices_unified:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.notices_unified").count())


In [0]:
from pyspark.sql import functions as F

# -----------------------------
# gold.dim_country
# -----------------------------
country_rows = [
    ("AUT", "Austria"), ("BEL", "Belgium"), ("BGR", "Bulgaria"),
    ("HRV", "Croatia"), ("CYP", "Cyprus"), ("CZE", "Czechia"),
    ("DNK", "Denmark"), ("EST", "Estonia"), ("FIN", "Finland"),
    ("FRA", "France"), ("DEU", "Germany"), ("GRC", "Greece"),
    ("HUN", "Hungary"), ("IRL", "Ireland"), ("ITA", "Italy"),
    ("LVA", "Latvia"), ("LTU", "Lithuania"), ("LUX", "Luxembourg"),
    ("MLT", "Malta"), ("NLD", "Netherlands"), ("POL", "Poland"),
    ("PRT", "Portugal"), ("ROU", "Romania"), ("SVK", "Slovakia"),
    ("SVN", "Slovenia"), ("ESP", "Spain"), ("SWE", "Sweden"),
    ("ISL", "Iceland"), ("LIE", "Liechtenstein"), ("NOR", "Norway"),
    ("CHE", "Switzerland"), ("GBR", "United Kingdom")
]

dim_country = spark.createDataFrame(country_rows, ["country_code", "country_name"])

dim_country.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.gold.dim_country"
)


# -----------------------------
# gold.dim_procedure_type
# -----------------------------
procedure_rows = [
    ("open", "Open procedure"),
    ("restricted", "Restricted procedure"),
    ("neg-wo-call", "Negotiated without prior call for competition"),
    ("neg-w-call", "Negotiated with prior call for competition"),
    ("comp-dial", "Competitive dialogue"),
    ("innovation", "Innovation partnership"),
    ("oth-single", "Other single-stage procedure"),
    ("oth-mult", "Other multiple-stage procedure"),
    ("direct", "Direct award"),
    ("unknown", "Unknown / not parsed")
]

dim_procedure_type = spark.createDataFrame(
    procedure_rows,
    ["procedure_type", "procedure_label"]
)

dim_procedure_type.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.gold.dim_procedure_type"
)


# -----------------------------
# gold.dim_buyer_type
# -----------------------------
buyer_type_rows = [
    ("body-pl", "Public law body"),
    ("body-pl-cga", "Central government authority"),
    ("body-pl-la", "Local authority"),
    ("body-pl-ra", "Regional authority"),
    ("body-pl-la-ra", "Local or regional authority"),
    ("eu-ins-bod-ag", "EU institution, body or agency"),
    ("grp-p-aut", "Group of public authorities"),
    ("pub-undert", "Public undertaking"),
    ("spec-rights-entity", "Entity with special or exclusive rights"),
    ("org-sub", "Organisation awarding subsidised contracts"),
    ("unknown", "Unknown / not parsed")
]

dim_buyer_type = spark.createDataFrame(
    buyer_type_rows,
    ["buyer_type", "buyer_type_label"]
)

dim_buyer_type.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.gold.dim_buyer_type"
)


# -----------------------------
# gold.dim_cpv
# CPV division labels from first two digits
# -----------------------------
cpv_division_rows = [
    ("03", "Agricultural, farming, fishing, forestry and related products"),
    ("09", "Petroleum products, fuel, electricity and other sources of energy"),
    ("14", "Mining, basic metals and related products"),
    ("15", "Food, beverages, tobacco and related products"),
    ("16", "Agricultural machinery"),
    ("18", "Clothing, footwear, luggage articles and accessories"),
    ("19", "Leather and textile fabrics, plastic and rubber materials"),
    ("22", "Printed matter and related products"),
    ("24", "Chemical products"),
    ("30", "Office and computing machinery, equipment and supplies"),
    ("31", "Electrical machinery, apparatus, equipment and consumables"),
    ("32", "Radio, television, communication, telecommunication and related equipment"),
    ("33", "Medical equipments, pharmaceuticals and personal care products"),
    ("34", "Transport equipment and auxiliary products"),
    ("35", "Security, fire-fighting, police and defence equipment"),
    ("37", "Musical instruments, sport goods, games, toys, handicraft, art materials"),
    ("38", "Laboratory, optical and precision equipments"),
    ("39", "Furniture, furnishings, domestic appliances and cleaning products"),
    ("41", "Collected and purified water"),
    ("42", "Industrial machinery"),
    ("43", "Machinery for mining, quarrying, construction equipment"),
    ("44", "Construction structures and materials"),
    ("45", "Construction work"),
    ("48", "Software package and information systems"),
    ("50", "Repair and maintenance services"),
    ("51", "Installation services"),
    ("55", "Hotel, restaurant and retail trade services"),
    ("60", "Transport services"),
    ("63", "Supporting and auxiliary transport services; travel agencies services"),
    ("64", "Postal and telecommunications services"),
    ("65", "Public utilities"),
    ("66", "Financial and insurance services"),
    ("70", "Real estate services"),
    ("71", "Architectural, construction, engineering and inspection services"),
    ("72", "IT services: consulting, software development, Internet and support"),
    ("73", "Research and development services and related consultancy services"),
    ("75", "Administration, defence and social security services"),
    ("76", "Services related to the oil and gas industry"),
    ("77", "Agricultural, forestry, horticultural, aquacultural and apicultural services"),
    ("79", "Business services: law, marketing, consulting, recruitment, printing and security"),
    ("80", "Education and training services"),
    ("85", "Health and social work services"),
    ("90", "Sewage, refuse, cleaning and environmental services"),
    ("92", "Recreational, cultural and sporting services"),
    ("98", "Other community, social and personal services")
]

dim_cpv = spark.createDataFrame(
    cpv_division_rows,
    ["cpv_division", "cpv_label"]
)

dim_cpv.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.gold.dim_cpv"
)


# -----------------------------
# Quick validation
# -----------------------------
for table in [
    "workspace.gold.dim_country",
    "workspace.gold.dim_procedure_type",
    "workspace.gold.dim_buyer_type",
    "workspace.gold.dim_cpv"
]:
    print(table, spark.table(table).count())

## Validation

In [0]:
print("Gold tables:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").show(truncate=False)

print("ML sanity checks:")
spark.sql(f"""
SELECT
  COUNT(*) AS ml_rows,
  ROUND(AVG(CASE WHEN is_single_bidder THEN 1 ELSE 0 END), 4) AS single_bidder_share,
  ROUND(AVG(CASE WHEN is_cross_border THEN 1 ELSE 0 END), 4) AS cross_border_share,
  COUNT(DISTINCT buyer_country) AS buyer_countries,
  COUNT(DISTINCT cpv_division) AS cpv_divisions
FROM {CATALOG}.{GOLD_SCHEMA}.ml_features
""").show(truncate=False)

print("Expected rough checks from handoff: ml_rows about 26,410, single_bidder_share about 0.454, cross_border_share about 0.024.")
print("If cross_border_share is 0, check Silver winner_country parsing before continuing to ML.")

display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.ml_features").limit(20))
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.daily_activity").orderBy("activity_date"))


Verification/Double-Checking structure and Count

In [0]:
%sql
SHOW TABLES IN workspace.gold;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.gold.ml_features;
SELECT AVG(CASE WHEN is_cross_border THEN 1 ELSE 0 END) FROM workspace.gold.ml_features;
SHOW TABLES IN workspace.gold;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.gold.ml_features;

In [0]:
%sql
SELECT AVG(CASE WHEN is_cross_border THEN 1 ELSE 0 END)
FROM workspace.gold.ml_features;

In [0]:
%sql
DESCRIBE TABLE workspace.gold.ml_contract_features;

SELECT COUNT(*)
FROM workspace.gold.ml_contract_features;